# FSM-based Pipeline Validation for RAG Systems
Notebook eksperimen untuk membandingkan **Baseline RAG** vs **RAG + FSM Validation**.

## 1) Install dependencies (jalankan sekali)

In [ ]:
%pip install -q langchain langchain-community transitions chromadb ragas google-generativeai datasets sentence-transformers

## 2) Import dan konfigurasi

In [ ]:
import os
import sys

# Menambahkan root directory ke sys.path agar bisa import dari folder src
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from datasets import load_dataset
from langchain.docstore.document import Document
from src.rag_baseline import build_baseline_rag, baseline_answer
from src.rag_with_fsm import fsm_rag_answer
import google.generativeai as genai

# Konfigurasi Path
DATA_DIR = "../data"
INDEX_DIR = "../index/chroma_db"
SQUAD_PATH = os.path.join(DATA_DIR, "squad_v2")

# Set API key
# os.environ['GEMINI_API_KEY'] = 'YOUR_KEY'
api_key = os.getenv('GEMINI_API_KEY')
if not api_key:
    print("WARNING: GEMINI_API_KEY belum diset. Pastikan os.environ['GEMINI_API_KEY'] sudah diisi.")
else:
    genai.configure(api_key=api_key)


## 3) Definisi FSM Validator

## 4) Baseline RAG

In [ ]:
def build_baseline_rag(documents, k=3):
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_documents(documents)

    embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
    vectorstore = Chroma.from_documents(chunks, embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={'k': k})
    return retriever

def baseline_answer(query, retriever, model_name='gemini-1.5-flash'):
    docs = retriever.get_relevant_documents(query)
    context = '\n'.join([d.page_content for d in docs])

    model = genai.GenerativeModel(model_name)
    prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer briefly and factually based on context."
    response = model.generate_content(prompt)

    return {
        'answer': response.text if hasattr(response, 'text') else '',
        'docs': docs,
        'context': context
    }

In [ ]:
def fsm_rag_answer(query, retriever):
    fsm = RAGStateMachine()
    result = {'answer': None, 'status': None, 'states_passed': [], 'context': '', 'docs': []}

    fsm.start()
    result['states_passed'].append('query_validation')
    if fsm.validate_query(query):
        fsm.query_valid()
    else:
        fsm.query_invalid()
        result['status'] = 'error'
        result['answer'] = 'Query tidak valid.'
        return result

    fsm.retrieve()
    docs = retriever.get_relevant_documents(query)
    result['states_passed'].append('retrieval_validation')
    if fsm.validate_retrieval(docs):
        fsm.docs_valid()
    else:
        fsm.docs_invalid()
        result['status'] = 'error'
        result['answer'] = 'Dokumen relevan tidak ditemukan.'
        return result

    context = '\n'.join([d.page_content for d in docs])
    result['states_passed'].append('context_validation')
    if fsm.validate_context(context):
        fsm.context_valid()
    else:
        fsm.context_invalid()
        result['status'] = 'error'
        result['answer'] = 'Konteks tidak valid.'
        return result

    fsm.generate()
    model = genai.GenerativeModel('gemini-1.5-flash')
    prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer briefly and factually based on context."
    response = model.generate_content(prompt)
    answer = response.text if hasattr(response, 'text') else ''
    result['states_passed'].append('generation')

    result['states_passed'].append('output_validation')
    if fsm.validate_output(answer):
        fsm.output_valid()
        result['status'] = 'success'
        result['answer'] = answer
    else:
        fsm.output_invalid()
        result['status'] = 'error'
        result['answer'] = 'Output tidak valid.'

    result['context'] = context
    result['docs'] = docs
    return result

## 6) Load dataset & siapkan dokumen

## 7) Jalankan eksperimen kecil (contoh 30 sampel)

In [ ]:
sample_n = 30
baseline_results = []
fsm_results = []

for item in dataset.select(range(sample_n)):
    query = item['question']

    baseline = baseline_answer(query, retriever)
    baseline_results.append({
        'question': query,
        'answer': baseline['answer'],
        'context': baseline['context'],
        'ground_truth': item['answers']['text'][0] if item['answers']['text'] else ''
    })

    fsm = fsm_rag_answer(query, retriever)
    fsm_results.append({
        'question': query,
        'answer': fsm['answer'],
        'context': fsm['context'],
        'ground_truth': item['answers']['text'][0] if item['answers']['text'] else '',
        'status': fsm['status'],
        'states_passed': fsm['states_passed']
    })

print('Baseline total:', len(baseline_results))
print('FSM total:', len(fsm_results))
print('FSM error count:', sum(1 for r in fsm_results if r['status'] == 'error'))

# Load dataset SQuAD v2 (menyimpan di folder data agar tidak download ulang)
dataset = load_dataset('squad_v2', split='validation[:500]', cache_dir=SQUAD_PATH)

documents = []
for item in dataset:
    if item['answers']['text']:
        documents.append(
            Document(
                page_content=item['context'],
                metadata={
                    'question': item['question'],
                    'answer': item['answers']['text'][0]
                }
            )
        )

print('Total documents:', len(documents))

# Build retriever (menggunakan fungsi dari src/rag_baseline.py)
retriever = build_baseline_rag(documents)


In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall
from datasets import Dataset

def eval_with_ragas(rows):
    data = {
        'question': [r['question'] for r in rows],
        'answer': [r['answer'] for r in rows],
        'contexts': [[r['context']] for r in rows],
        'ground_truth': [r['ground_truth'] for r in rows],
    }
    ds = Dataset.from_dict(data)
    return evaluate(ds, metrics=[faithfulness, answer_relevancy, context_recall])

baseline_score = eval_with_ragas(baseline_results)
fsm_success_rows = [r for r in fsm_results if r['status'] == 'success']
fsm_score = eval_with_ragas(fsm_success_rows) if fsm_success_rows else None

print('=== Baseline RAG ===')
print(baseline_score)
print('\n=== FSM RAG (success only) ===')
print(fsm_score)

sample_n = 30
baseline_results = []
fsm_results = []

for item in dataset.select(range(sample_n)):
    query = item['question']

    # Baseline RAG
    baseline = baseline_answer(query, retriever, api_key=api_key)
    baseline_results.append({
        'question': query,
        'answer': baseline,
        'context': "", # baseline_answer perlu diupdate jika ingin return context
        'ground_truth': item['answers']['text'][0] if item['answers']['text'] else ''
    })

    # FSM RAG
    fsm = fsm_rag_answer(query, retriever, api_key=api_key)
    fsm_results.append({
        'question': query,
        'answer': fsm['answer'],
        'context': fsm.get('context', ''),
        'ground_truth': item['answers']['text'][0] if item['answers']['text'] else '',
        'status': fsm['status'],
        'states_passed': fsm['states_passed']
    })

print('Baseline total:', len(baseline_results))
print('FSM total:', len(fsm_results))
print('FSM error count caught:', sum(1 for r in fsm_results if r['status'] == 'error'))
